# Module 4: Selection Mechanisms and What They Do to an Estimate

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

"Selection bias" is usually discussed as a single thing with a single
direction. It is not. **What was selected on determines both the sign and the
size of the bias, and what the estimator conditions on determines how much of
it survives.**

This module measures three selection rules in a world where the program does
not exist, so the bias each produces is the whole of the answer.

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Building a world with no program

The planted effect is known exactly: a factor of 0.88 on the rate, phased in
at 0, 25, 58, 83 and 100 percent over five months from July 2023. Dividing it
back out gives the counterfactual mean for every agency month, and a world
in which nothing was ever done.

In [ ]:
g = f.copy()
g["lo"] = np.log(g["n_arrests"])
pi = pd.PeriodIndex(g["year_month"], freq="M")
g["t"] = (pi.year.values - 2019) * 12 + pi.month.values - 1
g["yr"] = pi.year.values + (pi.month.values - 1) / 12.0

PHASE = {0: 0.0, 1: 0.25, 2: 0.58, 3: 0.83}
START = (2023 - 2019) * 12 + 6
w = np.array([PHASE.get(k, 1.0) if (a in TRAINED and k >= 0) else 0.0
              for a, k in zip(g["agency_id"], g["t"] - START)])
g["mu0"] = g["n_uof"].values / (0.88 ** w)

treated_post = (g["agency_id"].isin(TRAINED)) & (g["period"] == "after")
print(f"  counts at treated agencies after full implementation are inflated by "
      f"{100 * (g.loc[treated_post, 'mu0'].sum() / g.loc[treated_post, 'n_uof'].sum() - 1):.1f}%")
print("  which is the planted effect, removed")

## 3. Three ways to choose five agencies

In [ ]:
ids = sorted(g["agency_id"].unique())
pre = g[g["period"] == "before"]

level = {a: 100 * pre[pre["agency_id"] == a]["n_uof"].sum()
            / pre[pre["agency_id"] == a]["n_arrests"].sum() for a in ids}
slope = {}
for a in ids:
    s = pre[pre["agency_id"] == a]
    z = smf.glm("n_uof ~ yr", s, family=sm.families.Poisson(), offset=s["lo"]).fit()
    slope[a] = pct(z.params["yr"])
lw = g[(g["year_month"] >= "2023-01") & (g["year_month"] < "2023-07")]
recent = {a: 100 * lw[lw["agency_id"] == a]["n_uof"].sum()
             / lw[lw["agency_id"] == a]["n_arrests"].sum() for a in ids}

RULES = {"highest level before": lambda a: -level[a],
         "steepest downward trend before": lambda a: slope[a],
         "worst last six months before": lambda a: -recent[a]}
for name, key in RULES.items():
    print(f"  {name:32s} {', '.join(NAME[a].split()[0] for a in sorted(ids, key=key)[:5])}")

## 4. What each rule costs

In [ ]:
rng = np.random.default_rng(31)


def bias(pick, reps=200):
    s = g.copy()
    s["settled"] = ((s["agency_id"].isin(pick)) & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(pick)) & (s["period"] == "phase")).astype(float)
    out = []
    for _ in range(reps):
        s["y"] = rng.poisson(np.maximum(s["mu0"].values, 0.01))
        z = smf.glm("y ~ C(agency_id)+C(year_month)+settled+phase", s,
                    family=sm.families.Poisson(), offset=s["lo"]).fit()
        out.append(pct(z.params["settled"]))
    return np.array(out)


rows = []
for name, key in RULES.items():
    v = bias(sorted(ids, key=key)[:5])
    rows.append({"selection rule": name, "mean bias": f"{v.mean():+.2f}%",
                 "simulation sd": round(v.std(), 2)})
rand = []
for _ in range(60):
    rand.extend(bias(list(rng.choice(ids, 5, replace=False)), reps=6))
rand = np.array(rand)
rows.append({"selection rule": "at random", "mean bias": f"{rand.mean():+.2f}%",
             "simulation sd": round(rand.std(), 2)})
print("  the true effect in this world is 0.00 percent\n")
pd.DataFrame(rows).set_index("selection rule")

Four rules, four different answers, and the differences are not subtle.

**Selecting on the level costs +3.01 percent**, which understates a reduction.
Agency fixed effects absorb the level itself, so what is left is the part of a
high pre period average that was luck, and it has already partly reverted by
the time the post period starts.

**Selecting on the recent level costs +4.83 percent**, the largest bias here,
because six months of a small agency's rate is mostly noise and reverts hard.

**Selecting on the trend costs −1.94 percent**, and it is the dangerous one:
it biases **toward** finding an effect. That is what Summit County is, and it
is why Intermediate Module 8 had to remove it.

**Random assignment costs −0.11 percent** and is the only rule that is
unbiased. Note its spread of 8.7 against roughly 3 for the others: random
assignment buys freedom from bias, not precision.

## 5. The general statement

| Selected on | Direction of bias | Why |
|---|---|---|
| the **level** of the outcome | toward zero, if levels are conditioned on | mean reversion already partly spent |
| a **recent** level | toward zero, more strongly | a short window is mostly noise |
| the **trend** in the outcome | **toward the hypothesis** | the trend continues into the post period |
| something unrelated to the outcome | none | no backdoor path |

**The direction people assume, that selection flatters a program, holds only
for selection on trends.** Selection on levels, with a fixed effects
estimator, works the other way.

That is not a licence to relax about level based selection. It means the sign
of the bias has to be reasoned about in each case rather than assumed, and
that the estimator matters as much as the rule.

## Exercise

The estimator here conditions on agency fixed effects, which absorb the level.
Run the same three rules without them.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    def bias_nofe(pick, reps=120):
        s = g.copy()
        s["settled"] = ((s["agency_id"].isin(pick)) & (s["period"] == "after")).astype(float)
        s["phase"] = ((s["agency_id"].isin(pick)) & (s["period"] == "phase")).astype(float)
        s["tr"] = s["agency_id"].isin(pick).astype(float)
        out = []
        for _ in range(reps):
            s["y"] = rng.poisson(np.maximum(s["mu0"].values, 0.01))
            z = smf.glm("y ~ tr + C(year_month) + settled + phase", s,
                        family=sm.families.Poisson(), offset=s["lo"]).fit()
            out.append(pct(z.params["settled"]))
        return np.array(out)

    rows = []
    for name, key in RULES.items():
        pick = sorted(ids, key=key)[:5]
        rows.append({"selection rule": name,
                     "with agency fixed effects": f"{bias(pick, reps=120).mean():+.2f}%",
                     "without them": f"{bias_nofe(pick).mean():+.2f}%"})
    print("  the true effect in this world is 0.00 percent\n")
    display(pd.DataFrame(rows).set_index("selection rule"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Without agency fixed effects the biases change, and the level based rules
change most, because the level difference between the groups is no longer
absorbed and now enters the estimate directly.

**That is the point of the exercise.** "Selection bias" is not a property of
the selection rule alone. It is a property of the pair: what was selected on,
and what the estimator holds fixed. A rule that is harmless under one
specification is not harmless under another, and the only way to know is to
work out which backdoor paths the specification closes, which is
[Module 2](Module_02_DAGs_And_The_Backdoor_Criterion.ipynb).

The practical consequence for a report is that "agencies were selected on the
outcome" is not a sufficient description of the problem. The useful sentence
names the variable selected on, the estimator used, and the direction the
combination biases in.

</details>

---

**Next:** [Module 5: Two Way Fixed Effects Done Properly](Module_05_Two_Way_Fixed_Effects.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*